In [ ]:
import numpy as np
import torch
import torch.distributions as dist
from mfg import MFG, MFG_config
from solver import MajorAgentEstimator
from utility import make_example
from oracle import oracle_J
import matplotlib.pyplot as plt



In [ ]:
# ── Simulation grid ──────────────────────────────────────────────────────────
T   = 1       # time horizon (Section 6)
Ndt = 512    # time steps  → dt = 0.001

# ── Major bank parameters (Fig B.5, Fig B.6) ─────────────────────────────────
G        = 0.5   # relative market size of major bank  (F = 1-G = 0.5)
a        = 5     # minor bank mean-reversion rate (Section 6.2 baseline)
a_0      = a * G # = 2.5  market-clearing condition (eq 2.8): a_0 = a*G
sigma_0  = 1.0   # major bank reserve volatility
q_0      = 1.0   # major bank incentive to trade with central bank  (Fig B.5)
epslon_0 = 10.0  # major bank running penalty on reserve deviation  (Fig B.6)
c_0      = 0.0   # major bank terminal penalty                       (Fig B.6)

# ── Minor bank parameters (Fig B.5) ──────────────────────────────────────────
sigma    = 1.0   # minor bank reserve volatility
q        = 1.0   # minor bank incentive to trade with central bank
epslon   = 1.5   # minor bank running penalty  (must satisfy q^2 <= epslon)
c        = 0.0   # minor bank terminal penalty

# ── Monte Carlo ───────────────────────────────────────────────────────────────
N     = 256     # number of minor banks (Section 6)
N_sim = 300  # Monte Carlo paths     (Section 6)

cfg = MFG_config(
    T=T, Ndt=Ndt,
    a_0=a_0, sigma_0=sigma_0, c_0=c_0, epslon_0=epslon_0, q_0=q_0,
    a=a,     sigma=sigma,     c=c,     epslon=epslon,       q=q,
    G=G,
)
print(cfg)
mfg = MFG(cfg)
X, x_bar_obs, true_idx = make_example(mfg, N,seed=562)
# oracle_J(mfg, X,lam_entropy=20.0, true_major_idx=true_idx)


In [ ]:
print("\n=== EM relaxation (Tier 3, mean field unobserved -- estimated from X) ===")
est = MajorAgentEstimator(mfg, unknown=['G','a','q'], lam_entropy=50.0,temp_anneal=True, lr_E=0.1, lr_M=0.05,observed_xbar=None)
prob3, fitted3, step = est.fit(X, true_major_idx=true_idx, n_em_iters=20,
    n_inner_E_steps=5, n_inner_M_steps=5, verbose=True)
pred3 = int(prob3.argmax().item())
print(f"predicted={pred3}  true={true_idx}  correct={pred3==true_idx}  "
        f"G_hat={fitted3['G']:.4f}  G_true={cfg.G}")
print("top-3 prob:", np.round(np.sort(prob3.numpy())[::-1][:3], 3))
est.plot_loss(); est.plot_grad_norm(); est.plot_trajectory()

In [ ]:
xbar_emp, xbar_mf = est.estimated_xbar()
t_axis = np.linspace(0, mfg.T, mfg.Ndt + 1)[:-1]  # aligned to xbar_emp/xbar_mf (length Ndt)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(t_axis, xbar_emp, color='tab:blue', alpha=0.7,
        label='xbar_emp (weighted empirical mean)')
ax.plot(t_axis, xbar_mf, color='tab:orange', linewidth=2,
        label='xbar_mf (estimated, eq 4.5 @ fitted params)')
ax.plot(t_axis, x_bar_obs[:mfg.Ndt], color='black', linewidth=2, linestyle='--',
        label='xbar true (mfg.simulate)')
ax.set_xlabel('Time'); ax.set_ylabel('State value')
ax.set_title('Mean-field trajectory: estimated vs true')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# test

In [ ]:
import numpy as np
from mfg import MFG, MFG_config
from utility import make_example
from solver import MajorAgentEstimator,fit_G_a_plus_q
from oracle import oracle_J



# --- same baseline as experiment.py ---
A = 5
G_true = 0.5
a_0_true = A * G_true
SIGMA_0, Q_0, EPSLON_0, C_0 = 1.0, 1.0, 10.0, 0.0
SIGMA, Q, EPSLON, C         = 1.0, 1.0, 1.5, 0.0
T, NDT = 1, 512
N_BASE = 256
LAM_ENTROPY = 110

cfg = MFG_config(T=T, Ndt=NDT, a_0=a_0_true, sigma_0=SIGMA_0, c_0=C_0,
                  epslon_0=EPSLON_0, q_0=Q_0, a=A, sigma=SIGMA, c=C,
                  epslon=EPSLON, q=Q, G=G_true)
mfg = MFG(cfg)
mfg.solve_ODE()

seed = 66
X, x_bar_obs, true_idx = make_example(mfg, N_BASE, seed=seed)
# oracle_J(mfg, X, lam_entropy=LAM_ENTROPY, true_major_idx=true_idx)


In [ ]:

est = MajorAgentEstimator(mfg, unknown=['G', 'a','q', ],
                           lam_entropy=LAM_ENTROPY, lr_E=0.05, lr_M=0.05,temp_anneal=True)
prob, fitted, n_steps = est.fit(
    X, true_major_idx=true_idx, n_em_iters=150,
    n_inner_E_steps=15, n_inner_M_steps=5, verbose=True,
)

pred_idx = int(prob.argmax().item())
print(f"\npredicted={pred_idx}  true={true_idx}  correct={pred_idx==true_idx}  steps={n_steps}")
print("fitted:", fitted)
print("true:  ", {'G': G_true, 'a': A, 'a_0': a_0_true, 'q': Q, 'q_0': Q_0})

est.plot_loss(); est.plot_grad_norm(); est.plot_trajectory()


In [ ]:
import numpy as np
from mfg import MFG, MFG_config
from utility import make_example
from solver import fit_G_a_plus_q
from oracle import oracle_J
LAM_ENTROPY = 110
# --- baseline config (paper Section 6 regime) ---
A, G_TRUE = 5, 0.5
cfg = MFG_config(T=1, Ndt=512, a_0=A * G_TRUE, sigma_0=1.0, c_0=0.0, epslon_0=10.0,
                  q_0=1.0, a=A, sigma=1.0, c=0.0, epslon=1.5, q=5.0, G=G_TRUE)
mfg = MFG(cfg)
mfg.solve_ODE()

N_BASE = 256
seed = 2
X, x_bar_obs, true_idx = make_example(mfg, N_BASE, seed=seed)

prob, fitted, steps = fit_G_a_plus_q(mfg, X, true_major_idx=true_idx, n_em_iters=200,
    n_inner_E_steps=5, n_inner_M_steps=5, lam_entropy=110.0, verbose=True)

pred = int(prob.argmax().item())
print(f"\npredicted={pred}  true={true_idx}  correct={pred==true_idx}  steps={steps}")
print("fitted:", fitted)
print("true:   G=0.5  a=5  q=5  a+q=10")


In [ ]:

oracle_J(mfg, X, lam_entropy=LAM_ENTROPY, true_major_idx=true_idx)